# Coursework workflow

This notebook reruns the coursework cleaning and clustering procedure for source lineage. The three original notebooks and PDFs remain unchanged at the repository root. Their saved results are also recorded in `reports/historical/original_saved_text_outputs.json`.

The original clustering code constructed `X_cluster_scaled` from bed counts and `Amount`, but fitted K-Means and PCA to the seven-column `X_scaled` matrix. That matrix included ordinal facility, county, and variable codes and excluded `Amount`. This notebook retains the original matrix construction separately; notebooks 02 and 03 contain the updated analysis.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hospital_ml").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from hospital_ml.data import read_raw, prepare_data, write_json
SEED = 42


In [2]:
from hospital_ml.historical import run_historical
raw = read_raw()
replay = run_historical(raw, ROOT / "reports/historical", ROOT / "data/historical/original_cleaned.csv")
display(pd.DataFrame(replay["cleaning_steps"]))
print("Observed cleaned reporting rows:", replay["cleaned_rows"])
print("Exact eight-column match to shared cleaned file:", replay["values_match_original_shared_cleaned_eight_columns"])
print("Actual historical matrix:", replay["matrix_columns_actually_used"])
print("Replayed cluster sizes:", replay["cluster_sizes"])
print("PCA explained variance:", replay["pca_explained_variance_ratio"])

,step,rows,removed
0,raw,578350,NaN
1,drop_missing_selected_columns,577920,NaN
2,historical_projected_deduplication,553022,NaN
3,drop_non_numeric_values,542702,NaN
4,historical_sequential_iqr_BED_LIC,531435,11267.0
5,historical_sequential_iqr_BED_AVL,525845,5590.0
6,historical_sequential_iqr_BED_STF,513363,12482.0


Observed cleaned reporting rows: 513363
Exact eight-column match to shared cleaned file: True
Actual historical matrix: ['BED_LIC', 'BED_AVL', 'BED_STF', 'DAY_PER', 'FAC_NAME_ENC', 'COUNTY_ENC', 'Variable_ENC']
Replayed cluster sizes: {'0': 240344, '1': 170624, '2': 102395}
PCA explained variance: [0.4152355028530942, 0.17293325232285753]


## Reproduction limits

This run pins Python and library versions and explicitly sets `n_init='auto'` and PCA's full solver. The coursework environment and execution order were not recorded. Cluster labels can permute, and iterative results can vary across library versions.

The source-data check compares the selected analysis columns with the archived cleaned CSV when it is available. The original globally fitted preprocessing and projected deduplication remain isolated from the updated training and validation workflows.

In [3]:
print("Historical K-Means profile; Amount is mixed-unit and is not interpretable as dollars here")
display(pd.read_csv(ROOT / "reports/historical/replayed_cluster_means.csv"))
for limitation in replay["limitations"]:
    print("-", limitation)
print("For corrected capacity interpretation and facility-aware evaluation, use notebooks 02 and 03.")

Historical K-Means profile; Amount is mixed-unit and is not interpretable as dollars here


,legacy_cluster,BED_LIC,BED_AVL,BED_STF,Amount
0,0,68.601076,65.947754,41.772988,1.343318e+07
1,1,212.875838,205.316796,130.087971,3.017779e+07
2,2,415.233732,387.157976,255.618790,7.590176e+07


- Original library versions and executable order were not recorded.
- Original code cells have null execution counters but retained outputs.
- Global scaling and ordinal facility/county/variable codes are historical defects.
- Amount profile averages mix units and are retained only for audit, not interpretation.
For corrected capacity interpretation and facility-aware evaluation, use notebooks 02 and 03.
